In [1]:
import os # Used to create a structured output directory per abstraction level "k"
from dataclasses import dataclass   # Defines confirguration objects
from typing import Dict, List, Tuple # Helps in distinguising between event classes, macro activites, and abstraction levels(k)

import numpy as np # Supports efficient matrix operation for event-class corellation matrix
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram # Used to visualize event-class ehirarchy(dendogram)


# ========= pm4py Imports ==========
from pm4py.objects.log.obj import EventLog, Trace, Event
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as process_tree_converter
from pm4py.visualization.bpmn import visualizer as bpmn_visualizer

# ========= Clustering ==========
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.preprocessing import MinMaxScaler

# ========= Semantic Naming (optional) ==========
from sklearn.feature_extraction.text import TfidfVectorizer

# ========= Constants ==========
CASE_COL = "case:concept:name"
TS_COL = "time:timestamp"
ACTIVITY_COL = "activity"
CONCEPT_COL = "concept:name"

In [2]:
# test_vertical_clustering_no_pytest.py
import traceback
import numpy as np
import pandas as pd

from vertical_clustering import (
    CASE_COL, TS_COL, ACTIVITY_COL,
    SegmentationConfig,
    compute_event_class_correlation,
    build_event_class_hierarchy,
    get_mapping_for_k_clusters,
    apply_cluster_rewrite,
    collapse_consecutive_in_trace,
    adaptive_global_trace_segmentation,
    build_abstracted_logs,
)


# -----------------------------
# Minimal test runner utilities
# -----------------------------

class TestFailure(AssertionError):
    """
    PURPOSE
    -------
    Custom assertion error used by this no-pytest harness.

    INPUTS
    ------
    Inherits from AssertionError; instantiated with a message.

    RETURNS / USED BY
    -----------------
    Raised by assert_* helper functions and interpreted by run_test()
    as a test failure with a readable message.
    """
    pass

def assert_true(cond, msg="assertion failed"):
    """
    PURPOSE
    -------
    Minimal boolean assertion for this test harness.

    INPUTS
    ------
    cond : bool
        Condition expected to be True.
    msg : str
        Failure message if cond is False.

    RETURNS / USED BY
    -----------------
    Returns None.
    Raises TestFailure if cond is False.
    Used inside test_* functions to enforce invariants.
    """
    if not cond:
        raise TestFailure(msg)

def assert_equal(a, b, msg=None):
    """
    PURPOSE
    -------
    Minimal equality assertion for this test harness.

    INPUTS
    ------
    a, b : Any
        Values expected to be equal.
    msg : Optional[str]
        Custom failure message.

    RETURNS / USED BY
    -----------------
    Returns None.
    Raises TestFailure if a != b.
    Used inside test_* functions to validate exact outcomes.
    """
    if a != b:
        raise TestFailure(msg or f"Expected {b!r}, got {a!r}")

def assert_allclose(a, b, atol=1e-12, msg=None):
    """
    PURPOSE
    -------
    Assert that two numeric arrays are approximately equal.

    INPUTS
    ------
    a, b : array-like
        Arrays expected to be close elementwise.
    atol : float
        Absolute tolerance.
    msg : Optional[str]
        Custom failure message.

    RETURNS / USED BY
    -----------------
    Returns None.
    Raises TestFailure if np.allclose(...) fails.
    Used for correlation symmetry checks and other numeric invariants.
    """
    if not np.allclose(a, b, atol=atol):
        raise TestFailure(msg or "Arrays not close")

def assert_raises(exc_types, fn, *args, **kwargs):
    """
    exc_types: exception type or tuple of types
    
    PURPOSE
    -------
    Assert that calling a function raises a specific exception type.

    This supports the supervisor's "destructive testing" philosophy:
    we try to break a function with invalid inputs and confirm it fails loudly.

    INPUTS
    ------
    exc_types : Exception type or tuple[Exception types]
        The expected exception(s).
    fn : callable
        The function to call.
    *args, **kwargs
        Arguments forwarded to fn.

    RETURNS / USED BY
    -----------------
    Returns None on success (i.e., expected exception raised).
    Raises TestFailure if:
        - no exception is raised, or
        - a different exception is raised.
    Used throughout unit/integration tests for failure-seeking checks.
    
    """
    try:
        fn(*args, **kwargs)
    except exc_types:
        return
    except Exception as e:
        raise TestFailure(f"Expected {exc_types}, but got {type(e)}: {e}")
    raise TestFailure(f"Expected {exc_types} to be raised, but nothing was raised")

def run_test(name, fn):
    """
    PURPOSE
    -------
    Execute a single test function and print PASS/FAIL with a short traceback.

    INPUTS
    ------
    name : str
        Human-readable test label.
    fn : callable
        Zero-argument test function (e.g., test_corr_missing_columns).

    RETURNS / USED BY
    -----------------
    Returns bool:
        True  -> test passed
        False -> test failed
    Used by main() to run all tests and compute a summary.
    """
    try:
        fn()
        print(f"[PASS] {name}")
        return True
    except Exception as e:
        print(f"[FAIL] {name}: {type(e).__name__}: {e}")
        tb = traceback.format_exc(limit=3)
        print(tb)
        return False


# -----------------------------
# Toy data helpers
# -----------------------------

def ts(i: int) -> pd.Timestamp:
    """
    PURPOSE
    -------
    Generate a deterministic timestamp for toy event logs.

    INPUTS
    ------
    i : int
        Offset in seconds from a fixed base date.

    RETURNS / USED BY
    -----------------
    Returns pd.Timestamp.
    Used by make_case() to create ordered events with reproducible timestamps.
    """
    return pd.Timestamp("2020-01-01") + pd.Timedelta(seconds=i)

def make_case(case_id: str, acts, start=0):
    """
    PURPOSE
    -------
    Build one synthetic trace ("case") as a list of dict rows.

    INPUTS
    ------
    case_id : str
        Trace identifier used under CASE_COL.
    acts : iterable
        Activity labels to emit as events for this case.
    start : int
        Starting timestamp offset (seconds) for the first event.

    RETURNS / USED BY
    -----------------
    Returns List[dict], where each dict is one event row containing:
        CASE_COL, TS_COL, ACTIVITY_COL.
    Used by unit/integration tests to construct minimal logs.
    """
    return [{CASE_COL: case_id, TS_COL: ts(start+i), ACTIVITY_COL: a} for i, a in enumerate(acts)]

def df(rows):
    """
    PURPOSE
    -------
    Convert a list of row dicts into a pandas DataFrame.

    INPUTS
    ------
    rows : list[dict]
        Event rows.

    RETURNS / USED BY
    -----------------
    Returns pd.DataFrame.
    Used throughout tests as the standard input format for vertical_clustering.py.
    """
    return pd.DataFrame(rows)


# ============================================================
# Layer 1 — Unit tests
# ============================================================

def test_corr_missing_columns():
    """
    PURPOSE
    -------
    Failure-seeking unit test for compute_event_class_correlation():
    verify it fails loudly when required columns are missing.

    INPUTS
    ------
    None directly; constructs small DataFrames missing:
        - CASE_COL
        - TS_COL
        - label column (ACTIVITY_COL)

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1.
    """
    cfg = SegmentationConfig(window_size=3, attenuation=0.6)

    # Missing CASE_COL
    d1 = df([{TS_COL: ts(0), ACTIVITY_COL: "A"}])
    assert_raises((KeyError, ValueError), compute_event_class_correlation, d1, ACTIVITY_COL, cfg)

    # Missing TS_COL
    d2 = df([{CASE_COL: "1", ACTIVITY_COL: "A"}])
    assert_raises((KeyError, ValueError), compute_event_class_correlation, d2, ACTIVITY_COL, cfg)

    # Missing label column
    d3 = df([{CASE_COL: "1", TS_COL: ts(0), "other": "A"}])
    assert_raises((KeyError, ValueError), compute_event_class_correlation, d3, ACTIVITY_COL, cfg)

def test_corr_invalid_config():
    """
    PURPOSE
    -------
    Failure-seeking unit test for correlation scan config:
    invalid window_size / attenuation should raise ValueError.

    INPUTS
    ------
    None directly; creates a small valid log and passes invalid cfg:
        - window_size < 0
        - attenuation <= 0
        - attenuation > 1

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1.
    """
    base = df(make_case("1", ["A", "B", "C"], 0))

    assert_raises(ValueError, compute_event_class_correlation, base, ACTIVITY_COL,
                  SegmentationConfig(window_size=-1, attenuation=0.6))
    assert_raises(ValueError, compute_event_class_correlation, base, ACTIVITY_COL,
                  SegmentationConfig(window_size=3, attenuation=0.0))
    assert_raises(ValueError, compute_event_class_correlation, base, ACTIVITY_COL,
                  SegmentationConfig(window_size=3, attenuation=1.1))

def test_corr_empty_or_all_nan_behaviour():
    """
    PURPOSE
    -------
    Edge-case unit test for correlation scan:
    empty log or all-NaN labels should not crash.

    Acceptable outcomes (by design choice):
      - return empty (0x0) corr matrix and [] classes
      - OR fail loudly with ValueError

    INPUTS
    ------
    None directly; constructs:
        - empty DataFrame with correct columns
        - DataFrame where label column is entirely NaN

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1.
    """
    cfg = SegmentationConfig(window_size=3, attenuation=0.6)

    empty = pd.DataFrame(columns=[CASE_COL, TS_COL, ACTIVITY_COL])
    try:
        corr, classes = compute_event_class_correlation(empty, ACTIVITY_COL, cfg)
        assert_equal(corr.shape, (0, 0), "Expected empty corr matrix")
        assert_equal(classes, [], "Expected empty class list")
    except ValueError:
        pass  # allowed: "fail loudly"

    all_nan = df([
        {CASE_COL: "1", TS_COL: ts(0), ACTIVITY_COL: np.nan},
        {CASE_COL: "1", TS_COL: ts(1), ACTIVITY_COL: np.nan},
    ])
    try:
        corr, classes = compute_event_class_correlation(all_nan, ACTIVITY_COL, cfg)
        assert_equal(corr.shape, (0, 0))
        assert_equal(classes, [])
    except ValueError:
        pass

def test_corr_invariants_symmetry_finite_range():
    """
    PURPOSE
    -------
    Invariant-based unit test for compute_event_class_correlation():
    verify outputs satisfy paper-relevant properties:
        - corr is square
        - corr is symmetric
        - corr contains only finite values
        - corr is within [0,1] after normalization

    INPUTS
    ------
    None directly; constructs a small valid log.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1.
    """
    log = df(make_case("1", ["A", "B", "A", "C"], 0))
    cfg = SegmentationConfig(window_size=3, attenuation=0.6)
    corr, _ = compute_event_class_correlation(log, ACTIVITY_COL, cfg)

    assert_true(corr.shape[0] == corr.shape[1], "corr must be square")
    assert_allclose(corr, corr.T, atol=1e-12, msg="corr must be symmetric")
    assert_true(np.isfinite(corr).all(), "corr must be finite (no NaN/inf)")
    assert_true(np.min(corr) >= 0.0 and np.max(corr) <= 1.0, "corr must be in [0,1]")

def test_corr_no_cross_case_leakage():
    """
    PURPOSE
    -------
    Paper-faithful unit test:
    verify correlation scan does NOT create correlations across cases.

    Construct case1 with only A, case2 with only B.
    Expected: corr[A,B] == corr[B,A] == 0.

    INPUTS
    ------
    None directly; constructs a two-case log.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1.
    """
    rows = []
    rows += make_case("1", ["A", "A", "A"], 0)
    rows += make_case("2", ["B", "B", "B"], 100)
    log = df(rows)

    cfg = SegmentationConfig(window_size=3, attenuation=0.6)
    corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    idx = {c: i for i, c in enumerate(classes)}
    assert_equal(corr[idx["A"], idx["B"]], 0.0, "A/B must be 0 across independent cases")
    assert_equal(corr[idx["B"], idx["A"]], 0.0, "B/A must be 0 across independent cases")


# --- (Paper-faithful) NEW TEST 1: numeric extremes of attenuation ---
def test_corr_attenuation_extremes():
    """
    PURPOSE
    -------
    Paper-faithful stress test for the exponential attenuation rule:
    verify correlation scan remains stable for numeric extremes:
        - attenuation ~ 0 (near-underflow)
        - attenuation ~ 1 (almost no decay)

    Must still satisfy:
        - symmetry, finiteness, [0,1] range

    INPUTS
    ------
    None directly; builds a small multi-case log.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1 (paper Step 2.1).
    """
    # A small but non-trivial log with multiple classes and adjacency pairs.
    log = df(
        make_case("1", ["A", "B", "C", "A", "B"], 0) +
        make_case("2", ["B", "C", "A"], 100)
    )

    for a in [1e-12, 0.999999999999]:
        cfg = SegmentationConfig(window_size=10, attenuation=a)
        corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, cfg)

        assert_true(corr.shape[0] == corr.shape[1], "corr must be square")
        assert_allclose(corr, corr.T, atol=1e-12, msg=f"corr must be symmetric (attenuation={a})")
        assert_true(np.isfinite(corr).all(), f"corr must be finite (attenuation={a})")
        assert_true(np.min(corr) >= 0.0 and np.max(corr) <= 1.0, f"corr must be within [0,1] (attenuation={a})")

        # Sanity: with >=2 classes and adjacency in traces, corr should not be all zeros.
        if len(classes) >= 2:
            assert_true(corr.max() > 0.0, f"corr should have positive mass (attenuation={a})")


def test_hierarchy_breakers_and_invariants():
    """
    PURPOSE
    -------
    Failure-seeking + invariant test for build_event_class_hierarchy():
    confirm it fails loudly on invalid correlation matrices and produces
    a correctly-shaped linkage matrix Z on valid input.

    INPUTS
    ------
    None directly; constructs several inputs:
        - non-square corr
        - corr with n<2
        - corr containing NaN
        - valid corr from compute_event_class_correlation()

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1 (paper Step 2.2).
    """
    cfg = SegmentationConfig(linkage_method="complete")

    # Non-square should fail loudly (desired)
    assert_raises(ValueError, build_event_class_hierarchy, np.zeros((2, 3)), cfg)

    # n<2 should fail loudly (desired)
    assert_raises(ValueError, build_event_class_hierarchy, np.zeros((0, 0)), cfg)
    assert_raises(ValueError, build_event_class_hierarchy, np.zeros((1, 1)), cfg)

    # NaN should fail loudly (desired)
    corr_nan = np.array([[0.0, np.nan], [np.nan, 0.0]])
    assert_raises(ValueError, build_event_class_hierarchy, corr_nan, cfg)

    # Valid case -> Z shape
    log = df(make_case("1", ["A", "B", "A", "C"], 0))
    corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, SegmentationConfig(3, 0.6, "complete"))
    if len(classes) >= 2:
        Z = build_event_class_hierarchy(corr, SegmentationConfig(3, 0.6, "complete"))
        assert_equal(Z.shape, (len(classes) - 1, 4), "Z must be (n-1,4)")


def test_projection_functions():
    """
    PURPOSE
    -------
    Unit tests for projection + collapse steps (paper Step 2.3):
      - apply_cluster_rewrite fails when src_col missing
      - unmapped activities become -1 (your chosen behavior)
      - collapse removes only consecutive repeats within each case
      - collapse does not merge across cases

    INPUTS
    ------
    None directly; builds toy logs and mappings.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1 (paper Step 2.3).
    """
    log = df(make_case("1", ["A", "B", "A", "C"], 0))
    mapping = {"A": 0}

    # apply_cluster_rewrite missing src_col
    assert_raises(KeyError, apply_cluster_rewrite, log, mapping, "missing_col", "macro_activity_id")

    # unmapped -> -1
    out = apply_cluster_rewrite(log, mapping, ACTIVITY_COL, "macro_activity_id")
    assert_true((out.loc[out[ACTIVITY_COL].isin(["B", "C"]), "macro_activity_id"] == -1).all())

    # collapse invariants
    df2 = pd.DataFrame([
        {CASE_COL: "1", TS_COL: 1, "macro_activity_id": 0},
        {CASE_COL: "1", TS_COL: 2, "macro_activity_id": 0},
        {CASE_COL: "1", TS_COL: 3, "macro_activity_id": 1},
        {CASE_COL: "1", TS_COL: 4, "macro_activity_id": 1},
        {CASE_COL: "2", TS_COL: 1, "macro_activity_id": 9},
        {CASE_COL: "2", TS_COL: 2, "macro_activity_id": 9},
    ])
    collapsed = collapse_consecutive_in_trace(df2, "macro_activity_id")
    # no adjacent repeats per case
    for _, g in collapsed.sort_values([CASE_COL, TS_COL]).groupby(CASE_COL):
        seq = g["macro_activity_id"].tolist()
        assert_true(all(seq[i] != seq[i+1] for i in range(len(seq)-1)))

    # doesn't collapse across cases
    assert_equal(collapsed[CASE_COL].nunique(), 2)
    assert_equal(collapsed.shape[0], 3)


# --- (Paper-faithful) NEW TEST 2: interrupted episode collapse (uninterrupted runs only) ---
def test_collapse_interrupted_episodes_not_merged():
    """
    PURPOSE
    -------
    Paper-faithful unit test for the collapse rule:
    only *uninterrupted* episodes may be merged.

    Sequence 0,1,0 is an interrupted episode of 0 and must remain 0,1,0.

    INPUTS
    ------
    None directly; constructs a one-case macro log.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 1 (paper Step 2.3).
    """
    # Sequence: 0,1,0 must remain 0,1,0 after collapse (interrupted episode)
    df_in = pd.DataFrame([
        {CASE_COL: "1", TS_COL: 1, "macro_activity_id": 0},
        {CASE_COL: "1", TS_COL: 2, "macro_activity_id": 1},
        {CASE_COL: "1", TS_COL: 3, "macro_activity_id": 0},
    ])

    out = collapse_consecutive_in_trace(df_in, "macro_activity_id")
    assert_equal(out.shape[0], 3, "Interrupted episodes must not be merged across an intervening activity")
    assert_equal(out["macro_activity_id"].tolist(), [0, 1, 0], "Expected exact sequence 0,1,0 after collapse")


# ============================================================
# Layer 2 — Integration tests
# ============================================================

def test_integration_chain_and_determinism():
    """
    PURPOSE
    -------
    Integration test for the core chain:
        correlation → hierarchy → mapping

    Also checks determinism: running the same pipeline twice yields the same mapping.

    INPUTS
    ------
    None directly; constructs a small multi-case log.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 2.
    """
    log = df(make_case("1", ["A", "B", "A", "C"], 0) + make_case("2", ["A", "C", "B"], 100))
    cfg = SegmentationConfig(window_size=3, attenuation=0.7, linkage_method="complete")
    corr1, classes1 = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    Z1 = build_event_class_hierarchy(corr1, cfg)
    mapping1 = get_mapping_for_k_clusters(Z1, classes1, k=2)

    corr2, classes2 = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    Z2 = build_event_class_hierarchy(corr2, cfg)
    mapping2 = get_mapping_for_k_clusters(Z2, classes2, k=2)

    assert_equal(classes1, classes2)
    assert_equal(mapping1, mapping2)


# --- (Paper-faithful) NEW TEST 3: invalid k values when choosing abstraction level ---
def test_invalid_k_values_in_get_mapping_for_k_clusters():
    """
    PURPOSE
    -------
    Paper-faithful failure-seeking integration test:
    selecting an abstraction level k must reject invalid values:
        - k <= 0
        - k > number of event classes

    INPUTS
    ------
    None directly; constructs a log with >=3 classes, builds corr and hierarchy.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 2 (paper Step 2.3: choosing k).
    """
    log = df(
        make_case("1", ["A", "B", "C", "A"], 0) +
        make_case("2", ["B", "C", "A"], 100)
    )
    cfg = SegmentationConfig(window_size=3, attenuation=0.7, linkage_method="complete")

    corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    assert_true(len(classes) >= 3, "Need >=3 classes for invalid-k tests")

    Z = build_event_class_hierarchy(corr, cfg)
    n = len(classes)

    assert_raises(ValueError, get_mapping_for_k_clusters, Z, classes, 0)
    assert_raises(ValueError, get_mapping_for_k_clusters, Z, classes, -1)
    assert_raises(ValueError, get_mapping_for_k_clusters, Z, classes, n + 1)


def test_integration_golden_AB_vs_C():
    """
    PURPOSE
    -------
    “Golden” integration test with known structure:
    A and B always co-occur close; C appears separately.
    At k=2, we expect A and B to share a cluster and C to differ.

    INPUTS
    ------
    None directly; builds synthetic cases:
        - many ABAB traces
        - separate CCC traces

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as part of Layer 2.
    """
    rows = []
    for i in range(20):
        rows += make_case(f"ab_{i}", ["A", "B", "A", "B"], i * 100)
    for i in range(10):
        rows += make_case(f"c_{i}", ["C", "C", "C"], 5000 + i * 100)
    log = df(rows)

    cfg = SegmentationConfig(window_size=3, attenuation=0.9, linkage_method="complete")
    corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    Z = build_event_class_hierarchy(corr, cfg)
    mapping = get_mapping_for_k_clusters(Z, classes, k=2)

    # assert A and B in same cluster, C different
    assert_true(mapping["A"] == mapping["B"], "Expected A and B together at k=2")
    assert_true(mapping["C"] != mapping["A"], "Expected C separate from A/B at k=2")


# ============================================================
# Layer 3 — Application-level (end-to-end-ish, no BPMN)
# ============================================================

def test_app_smoke_build_abstracted_logs():
    """
    PURPOSE
    -------
    End-to-end-ish smoke test of the paper’s core pipeline across multiple k:
        correlation → hierarchy → mapping → rewrite → collapse

    Does not generate BPMNs (keeps tests fast and file-independent).

    INPUTS
    ------
    None directly; builds a small synthetic log and selects k_levels.

    RETURNS / USED BY
    -----------------
    Returns None.
    Called by main() as Layer 3 application smoke test.
    """
    rows = []
    rows += make_case("1", ["A", "B", "C", "A"], 0)
    rows += make_case("2", ["A", "C", "C", "B"], 100)
    rows += make_case("3", ["B", "A", "B"], 200)
    log = df(rows)

    cfg = SegmentationConfig(window_size=3, attenuation=0.7, linkage_method="complete")
    corr, classes = compute_event_class_correlation(log, ACTIVITY_COL, cfg)
    assert_true(len(classes) >= 2, "Need at least 2 classes for clustering")
    Z = build_event_class_hierarchy(corr, cfg)

    k_levels = [2, min(3, len(classes))]
    abstracted_logs, mappings = build_abstracted_logs(log, Z, classes, k_levels, do_collapse=True)

    for k in k_levels:
        assert_true(k in abstracted_logs and k in mappings)
        assert_true("macro_activity_id" in abstracted_logs[k].columns)
        assert_true(abstracted_logs[k]["macro_activity_id"].notna().all())


def main():
    """
    PURPOSE
    -------
    Run the full stress-test suite in a deterministic order and report results.

    INPUTS
    ------
    None.

    RETURNS / USED BY
    -----------------
    Returns None.
    Prints PASS/FAIL per test and a final summary.
    Exits with SystemExit(1) if any tests fail (useful in scripts/CI).
    """
    tests = [
        ("L1 corr missing columns", test_corr_missing_columns),
        ("L1 corr invalid config", test_corr_invalid_config),
        ("L1 corr empty/all-nan behavior", test_corr_empty_or_all_nan_behaviour),
        ("L1 corr invariants", test_corr_invariants_symmetry_finite_range),
        ("L1 corr no cross-case leakage", test_corr_no_cross_case_leakage),

        # Paper-faithful addition (Step 2.1)
        ("L1 corr attenuation extremes (paper)", test_corr_attenuation_extremes),

        ("L1 hierarchy breakers & invariants", test_hierarchy_breakers_and_invariants),
        ("L1 projection functions", test_projection_functions),

        # Paper-faithful addition (Step 2.3)
        ("L1 collapse interrupted episodes (paper)", test_collapse_interrupted_episodes_not_merged),

        ("L2 integration chain determinism", test_integration_chain_and_determinism),

        # Paper-faithful addition (Step 2.3 selection of abstraction level k)
        ("L2 invalid k values (paper)", test_invalid_k_values_in_get_mapping_for_k_clusters),

        ("L2 golden AB vs C", test_integration_golden_AB_vs_C),
        ("L3 app smoke build_abstracted_logs", test_app_smoke_build_abstracted_logs),
    ]

    passed = 0
    for name, fn in tests:
        if run_test(name, fn):
            passed += 1

    total = len(tests)
    print(f"\nSUMMARY: {passed}/{total} tests passed")

    # In Jupyter, you may prefer NOT to exit the kernel on failure.
    # If you want that behavior, keep the SystemExit. Otherwise, comment it out.
    if passed != total:
        raise SystemExit(1)


In [3]:
main()

[PASS] L1 corr missing columns
[PASS] L1 corr invalid config
[PASS] L1 corr empty/all-nan behavior
[PASS] L1 corr invariants
[PASS] L1 corr no cross-case leakage
[PASS] L1 corr attenuation extremes (paper)
[PASS] L1 hierarchy breakers & invariants
[PASS] L1 projection functions
[PASS] L1 collapse interrupted episodes (paper)
[PASS] L2 integration chain determinism
[PASS] L2 invalid k values (paper)
[PASS] L2 golden AB vs C
[PASS] L3 app smoke build_abstracted_logs

SUMMARY: 13/13 tests passed
